In [254]:
from game import Game
import time
import os
import re

## Load maps

In [255]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [256]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [257]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [258]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [259]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [260]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [261]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [262]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

### BFS

In [263]:
# TODO: Must return moves (if there is no solution return None), number of visited states
def solver_bfs(game_map):
    game = Game(game_map)

   
    init_player = game.get_player_position()  
    init_boxes = game.get_box_locations()    
    goals = game.get_goal_locations()        
    init_state = [init_player, init_boxes]  

 
    queue = []
    queue.append((init_player, init_boxes, ""))

    visited = []
    visited.append([init_player, init_boxes])

    expanded_count = 0

    while queue:
        current_player, current_boxes, current_path = queue.pop(0)
        expanded_count += 1

   
        if game.is_game_won():
            return current_path, expanded_count

        game.set_player_position(current_player)
        game.set_box_positions(current_boxes)

        for move in ["U", "D", "L", "R"]:
            result = game.apply_move(move)
            if result:
                new_player = game.get_player_position()
                new_boxes = game.get_box_locations()

                new_state = [new_player, new_boxes]


                if not is_state_visited(new_state, visited):
                    visited.append(new_state)
                    new_path = current_path + move
                    queue.append((new_player, new_boxes, new_path))

            game.set_player_position(current_player)
            game.set_box_positions(current_boxes)

    return None, expanded_count


def is_state_visited(new_state, visited):
    
    for state in visited:
        if state[0] == new_state[0] and state[1] == new_state[1]:
            return True
    return False



### DFS

In [264]:
# TODO: Must return moves, number of visited states
def solver_dfs(game_map):
    game = Game(game_map)

 
    init_player = game.get_player_position() 
    init_boxes = game.get_box_locations()   
    goals = game.get_goal_locations()      
    init_state = [init_player, init_boxes]  
 
    stack = []
    stack.append((init_player, init_boxes, ""))

   
    visited = []
    visited.append([init_player, init_boxes])

    expanded_count = 0

    while stack:
  
        current_player, current_boxes, current_path = stack.pop()
        expanded_count += 1

       
        if game.is_game_won():
            return current_path, expanded_count

      
        game.set_player_position(current_player)
        game.set_box_positions(current_boxes)

    
        for move in ["U", "D", "L", "R"]:
            result = game.apply_move(move)
            if result:
                new_player = game.get_player_position()
                new_boxes = game.get_box_locations()

          
                new_state = [new_player, new_boxes]

                if not is_state_visited(new_state, visited):
                    visited.append(new_state)
                    new_path = current_path + move
                    stack.append((new_player, new_boxes, new_path))

            
            game.set_player_position(current_player)
            game.set_box_positions(current_boxes)

    return None, expanded_count


def is_state_visited(new_state, visited):
  
    for state in visited:
        if state[0] == new_state[0] and state[1] == new_state[1]:
            return True
    return False



### IDS

In [265]:
# TODO: Must return moves, number of visited states
def solver_ids(game_map):
    game = Game(game_map)

    init_player = game.get_player_position()  
    init_boxes  = game.get_box_locations()     
    goals       = game.get_goal_locations() 

    init_state = [init_player, init_boxes]

    expanded_count = 0

    max_depth = 25

    def depth_limited_search(state, current_depth, depth_limit, path, visited):

        nonlocal expanded_count

        player_pos = state[0]
        boxes_pos  = state[1]

        solved = True
        for i in range(len(boxes_pos)):
            if boxes_pos[i] != goals[i]:
                solved = False
                break
        if solved:
            return path 
        
        if current_depth == depth_limit:
            return None
        
        game.set_player_position(player_pos)
        game.set_box_positions(boxes_pos)

        expanded_count += 1

        for move in ["U", "D", "L", "R"]:
            result = game.apply_move(move)
            if result:
                new_player = game.get_player_position()
                new_boxes  = game.get_box_locations()
                new_state  = [new_player, new_boxes]

                if new_state not in visited:
                    visited.append(new_state)
                    new_path = path + move
                    solution = depth_limited_search(new_state,
                                                    current_depth + 1,
                                                    depth_limit,
                                                    new_path,
                                                    visited)
                    
                    game.set_player_position(player_pos)
                    game.set_box_positions(boxes_pos)

                    if solution is not None:
                        return solution
                else:
                    game.set_player_position(player_pos)
                    game.set_box_positions(boxes_pos)
            else:
                game.set_player_position(player_pos)
                game.set_box_positions(boxes_pos)
        return None
    
    for depth in range(max_depth + 1):
        visited = [init_state] 
        solution = depth_limited_search(init_state, 0, depth, "", visited)

        if solution is not None:
            return solution, expanded_count
    return None, expanded_count

### A*

In [266]:
# TODO
def heuristic1(game):
   
    player = game.get_player_position()
    boxes = game.get_box_locations()
    goals = game.get_goal_locations()

    total_cost = 0

    for i in range(len(boxes)):
        (rB, cB) = boxes[i]
        (rG, cG) = goals[i]
        dist_box_goal = abs(rB - rG) + abs(cB - cG)
        total_cost += dist_box_goal

    if len(boxes) > 0:
        (rP, cP) = player
        (rB, cB) = boxes[0]
        dist_pb = abs(rP - rB) + abs(cP - cB)
        total_cost +=dist_pb

    return total_cost

def heuristic2(game):
   
    player = game.get_player_position()
    boxes = game.get_box_locations()
    goals = game.get_goal_locations()

    total_cost = 0

    for i in range(len(boxes)):
        (rB, cB) = boxes[i]
        (rG, cG) = goals[i]
        dist_box_goal = abs(rB - rG) + abs(cB - cG)
        total_cost += dist_box_goal

    if len(boxes) > 0:
        (rP, cP) = player
        min_dist_pb = float('inf')
        for box in boxes:
            (rB, cB) = box
            dist_pb = abs(rP - rB) + abs(cP - cB)
            if dist_pb < min_dist_pb:
                min_dist_pb = dist_pb
        total_cost += min_dist_pb

    return total_cost


def solver_astar(game_map, heuristic_func=heuristic2, weight=2):
    game = Game(game_map)

    init_player = game.get_player_position()
    init_boxes = game.get_box_locations()
    goals = game.get_goal_locations()

    queue = []
    visited = set()

    g_init = 0
    h_init = heuristic_func(game)
    f_init = g_init + weight * h_init

    queue.append((f_init, g_init, init_player, tuple(init_boxes), ""))
    visited.add((init_player, tuple(init_boxes)))

    expanded_count = 0

    while queue:
        min_index = 0
        min_f_val = queue[0][0]
        for i in range(1, len(queue)):
            if queue[i][0] < min_f_val:
                min_f_val = queue[i][0]
                min_index = i

        curr_f, curr_g, curr_pl, curr_boxes, curr_path = queue.pop(min_index)
        expanded_count += 1

        game.set_player_position(curr_pl)
        game.set_box_positions(list(curr_boxes))

        if game.is_game_won():
            return curr_path, expanded_count

        for move in ["U", "D", "L", "R"]:
            result = game.apply_move(move)
            if result:
                new_pl = game.get_player_position()
                new_boxes = game.get_box_locations()

                new_state = (new_pl, tuple(new_boxes))

                if new_state not in visited:
                    visited.add(new_state)

                    new_g = curr_g + 1
                    new_h = heuristic_func(game)
                    new_f = new_g + weight * new_h

                    new_path = curr_path + move

                    queue.append((new_f, new_g, new_pl, tuple(new_boxes), new_path))

                game.set_player_position(curr_pl)
                game.set_box_positions(list(curr_boxes))
            else:
                game.set_player_position(curr_pl)
                game.set_box_positions(list(curr_boxes))

    return None, expanded_count


## Solve

In [267]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
    "A*": solver_astar
}

In [268]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [274]:
solve(5, "BFS") # Solve map 1 using BFS

BFS took 32.92 seconds on map 5 and visited 6329 states.
15 moves were used: ULDDRDLLLUUURUR


In [270]:
"""
def solve_all():
    for map in range(min(map_file_indices), max(map_file_indices) + 1):
        for method in SOLVERS.keys():
            solve(map, method)
  """          

'\ndef solve_all():\n    for map in range(min(map_file_indices), max(map_file_indices) + 1):\n        for method in SOLVERS.keys():\n            solve(map, method)\n  '

In [271]:
solve_all() # Solve all maps using all methods

NameError: name 'solve_all' is not defined